# 03 — Experiment Results

## Where results live

Each run appends to `outputs/ledger/<run>.jsonl` — one JSON line per node. A
sidecar `outputs/ledger/<run>.manifest.json` records the data-selection and
search-contract knobs (split seed, fractions, allowed levers, eta, …).

This notebook loads a ledger and lets you:
- Browse all nodes as a DataFrame
- Inspect any individual node (params, spec_config, scores)
- Visualise the search tree and (for search runs) the Ladder climb

**In the Setup cell, set `RUN` to a run name under `outputs/ledger/` (no
extension), e.g. `"2026-06-20-baseline"`.** The path is resolved against the
repo's `outputs/` dir, so it works from any working directory. If the run is
missing the cell lists the runs that *are* available and stops — it will not
fabricate data. Use `RUN = "demo"` for a synthetic example.

> Note: baseline/sweep runs are flat comparisons (no `val` queries), so the
> Ladder-climb and accepted/rejected views stay empty — those are for adaptive
> *search* runs.

In [ ]:
import os, warnings
import matplotlib.pyplot as plt
import pandas as pd

from scrye.config import OUTPUTS_DIR
from scrye.ledger import Ledger
from scrye.tree_view import tree_lines
from scrye.results import (
    nodes_frame, node_report, format_node_report,
    best_progression, spec_from_node, replay_proposals,
)
from scrye.search import GatePolicy, Proposal, SearchContract, run_search
from scrye.spec import PipelineSpec

# --- PARAMETERS (edit this) ---
RUN = "2026-06-20-baseline"   # a run name under outputs/ledger/ (no extension).
                              # Use RUN = "demo" for a synthetic example.
# -------------------------------

# Resolve against the repo's outputs/ dir (absolute) so this works no matter
# which directory the kernel runs in — Jupyter often runs from notebooks/, not
# the repo root, which is why a bare relative path silently missed the file.
LEDGER_DIR = OUTPUTS_DIR / "ledger"
RUN_PATH = str(LEDGER_DIR / f"{RUN}.jsonl")

if not os.path.exists(RUN_PATH):
    if RUN != "demo":
        # Do NOT fabricate a demo at a real run's path — fail loudly instead.
        avail = sorted(p.stem for p in LEDGER_DIR.glob("*.jsonl"))
        raise FileNotFoundError(
            f"No ledger at {RUN_PATH}.\n"
            f"Available runs in {LEDGER_DIR}: {avail or '(none)'}\n"
            f"Set RUN to one of those (no extension), or RUN = 'demo' for a synthetic example."
        )
    # RUN == 'demo': generate a tiny deterministic demo ledger. This branch only
    # runs when you explicitly ask for the demo, so it never masks a real run.
    print("[DEMO] Generating a synthetic demo ledger with a stub score_fn.\n")
    LEDGER_DIR.mkdir(parents=True, exist_ok=True)
    _proposals_list = [
        Proposal("recalib.global_temp", {"T": 1.5}, "Flatten over-confident instruct distributions"),
        Proposal("recalib.dirichlet", {"alpha": 0.05}, "Add smoothing toward uniform to lift under-predicted tails"),
    ]
    _dev_scores = [42.3, 45.1]
    _val_scores = [43.8, 46.5]
    _call_counts = {"dev": 0, "val": 0}

    def _stub_score_fn(spec, recs):
        if recs == ["val"]:
            i = min(_call_counts["val"], len(_val_scores) - 1)
            _call_counts["val"] += 1
            return _val_scores[i], {"cost_usd": 0.001}
        i = min(_call_counts["dev"], len(_dev_scores) - 1)
        _call_counts["dev"] += 1
        return _dev_scores[i], {"cost_usd": 0.002}

    _it = iter(_proposals_list)
    _demo_ledger = Ledger(RUN_PATH)
    _contract = SearchContract(
        goal="Improve SimBench S score on dev+val",
        allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
        autonomy_budget=10,
        val_query_budget=5,
        gate_policy=GatePolicy(gate_on_surprise=False),
        eta=0.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run_search(
            _contract, _demo_ledger,
            dev=["dev"], val=["val"],
            score_fn=_stub_score_fn,
            propose_fn=lambda state: next(_it, None),
            now_fn=lambda: "2026-06-20T00:00:00",
        )
    from scrye.manifest import RunManifest, dataset_fingerprint
    from scrye.levers import REGISTRY_VERSION
    RunManifest(
        registry_version=REGISTRY_VERSION, split_seed=0,
        split_fractions={"dev": 0.5, "val": 0.25, "test": 0.25}, split_unit="question",
        pin_required_to="test", goal=_contract.goal, allowed_levers=_contract.allowed_levers,
        autonomy_budget=_contract.autonomy_budget, val_query_budget=_contract.val_query_budget,
        eta=_contract.eta, cost_cap_usd=1000.0, model="gemini-flash-lite",
        temperature=0.0, seed=0, dataset_fingerprint=dataset_fingerprint([]),
        created_at="2026-06-20T00:00:00",
    ).save(RUN_PATH.replace(".jsonl", ".manifest.json"))
    print(f"[DEMO] Wrote {RUN_PATH} ({len(_demo_ledger.nodes)} nodes) + manifest.\n")

print(f"Using ledger: {RUN_PATH}")


In [ ]:
ledger = Ledger.load(RUN_PATH)
print(f"Loaded {len(ledger.nodes)} nodes, global_k={ledger.global_k()}, "
      f"best_val={ledger.best_val():.4f}, total_cost=${ledger.total_cost():.4f}\n")
print("Search tree:")
print("\n".join(tree_lines(ledger)))


In [ ]:
df = nodes_frame(ledger)
display(df)

# Scatter: dev_score vs val_score, coloured by accepted status
val_rows = df[df["val_score"].notna()].copy()
if not val_rows.empty:
    colors = val_rows["accepted"].map({True: "tab:green", False: "tab:red", None: "tab:gray"})
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(val_rows["dev_score"], val_rows["val_score"], c=colors, s=80, edgecolors="k", linewidths=0.5)
    ax.set_xlabel("Dev score")
    ax.set_ylabel("Val score")
    ax.set_title("Dev vs Val score (green=accepted, red=rejected)")
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="tab:green", label="accepted"),
        Patch(facecolor="tab:red", label="rejected"),
        Patch(facecolor="tab:gray", label="no val query"),
    ]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes to plot.")


In [ ]:
# Change NODE_ID to inspect a different node
NODE_ID = ledger.nodes[-1].node_id  # last node by default

print(format_node_report(ledger, NODE_ID))


In [ ]:
bp = best_progression(ledger)
display(bp)

if not bp.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(bp["step"], bp["best_so_far"], marker="o", color="steelblue", label="best_so_far")
    ax.scatter(bp["step"], bp["val_score"], color="lightgray", zorder=3, label="val_score (all)")
    # Mark new-best points
    new_best = bp[bp["is_new_best"]]
    ax.scatter(new_best["step"], new_best["best_so_far"], color="gold",
               s=120, zorder=4, edgecolors="darkorange", linewidths=1.5, label="new best")
    ax.set_xlabel("Step (val query index)")
    ax.set_ylabel("Val score")
    ax.set_title("Ladder climb — best val score over search steps")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes for progression plot.")


## Reproducibility

### How to reproduce this run

The sidecar `<run>.manifest.json` records every knob that was deterministic at search time:
split seed/fractions/unit, allowed levers, eta, model, temperature, and a dataset fingerprint.

To reproduce:

1. **Re-select the data** — call `make_split(records, seed=manifest.split_seed, fractions=manifest.split_fractions, unit=manifest.split_unit, pin_required_to=manifest.pin_required_to)` to get identical dev/val/test sets.
2. **Replay the proposals** — `replay_proposals(ledger)` returns the exact sequence of `Proposal(lever_id, params, rationale)` objects recorded in the ledger.
3. **Re-run search** — pass a fresh `Ledger`, the same `SearchContract`, and a `propose_fn` that exhausts the replayed list.

**Why it's identical:** cached LLM calls (temp=0 + on-disk cache) are byte-for-byte repeatable. Uncached calls with temp=0 are near-identical but not guaranteed by the API. The split and proposal sequence are pure functions of the recorded seeds.

The cell below runs an **offline reproducibility check** using the same stub score function and asserts that the replayed ledger matches the original.

In [ ]:
# Load manifest if present
import json as _json
manifest_path = RUN_PATH.replace(".jsonl", ".manifest.json")
if os.path.exists(manifest_path):
    from scrye.manifest import RunManifest
    print("Run manifest:")
    for k, v in _json.loads(open(manifest_path).read()).items():
        print(f"  {k}: {v}")
else:
    print("(No manifest file found.)")

print()

# The offline replay check below uses a deterministic STUB scorer, so it only
# applies to the synthetic 'demo' run. For real runs, reproducibility comes from
# the on-disk LLM cache (temp=0) plus the seeded split/proposal sequence — see
# the manifest above and `replay_proposals(ledger)`.
if RUN != "demo":
    print(f"\nOffline replay check is demo-only; skipping for real run '{RUN}'.")
    print("Reproducibility for real runs: cached temp=0 LLM calls + seeded split/proposals.")
else:
    _replayed_proposals = replay_proposals(ledger)
    _replay_iter = iter(_replayed_proposals)
    _rc = {"dev": 0, "val": 0}
    _dscores = [42.3, 45.1]
    _vscores = [43.8, 46.5]

    def _replay_score_fn(spec, recs):
        if recs == ["val"]:
            i = min(_rc["val"], len(_vscores) - 1)
            _rc["val"] += 1
            return _vscores[i], {"cost_usd": 0.001}
        i = min(_rc["dev"], len(_dscores) - 1)
        _rc["dev"] += 1
        return _dscores[i], {"cost_usd": 0.002}

    import tempfile as _tmp
    _replay_ledger = Ledger(os.path.join(_tmp.mkdtemp(), "replay.jsonl"))
    _replay_contract = SearchContract(
        goal="Improve SimBench S score on dev+val",
        allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
        autonomy_budget=10,
        val_query_budget=5,
        gate_policy=GatePolicy(gate_on_surprise=False),
        eta=0.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run_search(
            _replay_contract, _replay_ledger,
            dev=["dev"], val=["val"],
            score_fn=_replay_score_fn,
            propose_fn=lambda state: next(_replay_iter, None),
            now_fn=lambda: "2026-06-20T00:00:00",
        )

    orig_levers = [n.lever_id for n in ledger.nodes]
    replay_levers = [n.lever_id for n in _replay_ledger.nodes]
    assert orig_levers == replay_levers, f"Lever mismatch: {orig_levers} != {replay_levers}"
    assert ledger.global_k() == _replay_ledger.global_k(), (
        f"global_k mismatch: {ledger.global_k()} != {_replay_ledger.global_k()}"
    )
    print(f"Original  : levers={orig_levers}, global_k={ledger.global_k()}")
    print(f"Replayed  : levers={replay_levers}, global_k={_replay_ledger.global_k()}")
    print("REPRODUCED ✓")


---
## Cross-experiment analysis

The plots below read the saved runs in `outputs/runs/` and show the **grouped**
SimBench score — the demographic-conditioning task, where method differences
actually show. (Pop is mostly unconditioned, so methods bunch there.)


In [ ]:
import io, json
import numpy as np
from matplotlib.figure import Figure
from scrye import evaluate as ev
from scrye import viz

RUNS_DIR = OUTPUTS_DIR / "runs"

# Render returned Figure objects inline (same trick as 00_starter).
def _fig_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    return buf.getvalue()
get_ipython().display_formatter.formatters["image/png"].for_type(Figure, _fig_png)

def grouped_table(run):
    """Per-system grouped SimBench score (mean + 95% CI) from a saved run."""
    per = json.load(open(RUNS_DIR / f"{run}.results.json"))
    rows = []
    for key, recs in per.items():
        sb = ev.score_by(pd.DataFrame(recs), "split")
        g = sb[sb["split"] == "grouped"]
        if len(g):
            rows.append({"key": key,
                         "mean_score": float(g.iloc[0]["mean_score"]),
                         "ci_low": float(g.iloc[0]["ci_low"]),
                         "ci_high": float(g.iloc[0]["ci_high"])})
    return pd.DataFrame(rows)


### Stage 01 — conditioning strategies (gemini-2.5-flash-lite)

In [ ]:
t1 = grouped_table("2026-06-20-baseline").rename(columns={"key": "system"})
display(t1.sort_values("mean_score", ascending=False).round(2).reset_index(drop=True))
f = viz.plot_system_comparison(t1, label_col="system")
f.axes[0].set_title("Stage 01 — conditioning strategy (grouped SimBench S)")
f


### Stage 02 — model portability (top strategies × models)

In [ ]:
t2 = grouped_table("2026-06-20-model-sweep")
t2[["model", "system"]] = t2["key"].str.split("::", expand=True)
order_m = ["gemini-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
piv = t2.pivot(index="system", columns="model", values="mean_score")
piv = piv[[m for m in order_m if m in piv.columns]]
display(piv.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
systems = piv.index.tolist(); x = np.arange(len(systems)); w = 0.25
for i, m in enumerate(piv.columns):
    ax.bar(x + (i - 1) * w, piv[m].values, w, label=m.replace("gemini-", ""))
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel("grouped SimBench S"); ax.axhline(0, color="0.6", ls="--", lw=1)
ax.set_title("Stage 02 — model portability (grouped)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


### Stage 03 — persona mechanisms (gemini-3.1-flash-lite)

In [ ]:
t3 = grouped_table("2026-06-20-persona-mechanisms").rename(columns={"key": "system"})
display(t3.sort_values("mean_score", ascending=False).round(2).reset_index(drop=True))
f = viz.plot_system_comparison(t3, label_col="system")
f.axes[0].set_title("Stage 03 — persona mechanism (grouped SimBench S)")
f


### Does the original *faithful* baseline benefit from a better model?

**Yes.** This is the `faithful` row of the Stage 02 sweep — the original SimBench
first-person persona, held fixed, across model generations.


In [ ]:
fa = t2[t2["system"] == "faithful"].set_index("model").reindex(order_m)
display(fa[["mean_score", "ci_low", "ci_high"]].round(2))

fig, ax = plt.subplots(figsize=(7, 4))
xm = np.arange(len(fa)); means = fa["mean_score"].to_numpy()
err = np.vstack([means - fa["ci_low"].to_numpy(), fa["ci_high"].to_numpy() - means])
ax.bar(xm, means, 0.6, yerr=err, capsize=4, color="#C44E52", alpha=0.85)
ax.set_xticks(xm); ax.set_xticklabels([m.replace("gemini-", "") for m in fa.index])
ax.set_ylabel("grouped SimBench S"); ax.axhline(0, color="0.6", ls="--", lw=1)
ax.set_title("Original 'faithful' first-person baseline, across models")
for xi, m in zip(xm, means):
    ax.annotate(f"{m:.1f}", (xi, m), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print("The naive first-person baseline nearly doubles its grouped score just by "
      "moving to gemini-3.1-flash-lite — the better model recovers most of the gap.")


---
## Validation lineage — and the system that emerges

We iterated entirely on `dev`, promoting a new system only when it cleared a
gated **val** confirmation (Stages 05 / 12 / 16). The table and charts below show
held-out **validation** performance for the original `faithful` baseline and every
val-confirmed winner. Two levers separate bottom from top — the **model jump**
(2.5→3.1-flash-lite, un-gated) and our **method** (the gated lineage).

The last promotion, the **task-kind router** (Stage 16), won val by +1.1 pooled.
But — getting ahead to the one-shot test below — that final edge **did not
transfer**: on test the router and the simpler **`calibrated_commitment` +
abstain** (Stage 12) are statistically tied. So the deployable recommendation
leads with the **simpler `cc+abstain`**; the router is an interesting idea whose
val lift didn't replicate (see the test section).

> Numbers are the recorded **gated val** evaluations (faithful references +
> Stages 05 / 12 / 16). The notebook does **not** re-run val — val is touched
> only at gated confirmations (leakage discipline). The one-shot **test** number
> is in the final section. For repeatable per-record diagnostics, use
> `scrye.results.best_analysis_run()` (a **dev** run — never val/test).


In [ ]:
# Recorded held-out VAL scores. All @ gemini-3.1-flash-lite unless the row says 2.5.
# Provenance: faithful references + Stage 05 (anti_flattening), Stage 12
# (cc+abstain), Stage 16 (router). Val is NOT re-run here — it is touched only at
# gated confirmations (see docs/experiments/stage-{05,12,16}-*.md).
val_lineage = pd.DataFrame([
    ["0a baseline", "faithful @ 2.5-flash-lite",       30.23, 22.90, 26.60, "original baseline"],
    ["0b model",    "faithful @ 3.1-flash-lite",       46.98, 26.61, 36.90, "model jump (un-gated)"],
    ["1 Stage 05",  "anti_flattening",                 53.02, 33.64, 43.43, "✓ vs faithful"],
    ["2 Stage 12",  "calibrated_commitment + abstain", 53.68, 36.90, 45.38, "✓ vs anti_flattening"],
    ["3 Stage 16",  "task-kind router (FINAL)",        55.57, 38.27, 47.01, "✓ vs cc+abstain"],
], columns=["step", "system", "grouped", "pop", "pooled", "gate"])
display(val_lineage.set_index("step").round(2))

g0, p0, o0 = 46.98, 26.61, 36.90   # faithful @ 3.1 (method baseline, model held fixed)
g1, p1, o1 = 55.57, 38.27, 47.01   # final router
print("METHOD lever — faithful(@3.1) -> final router on held-out val:")
print(f"  grouped +{g1-g0:.2f}   pop +{p1-p0:.2f}   pooled +{o1-o0:.2f}")
print("MODEL lever — faithful 2.5 -> 3.1-flash-lite:")
print(f"  grouped +{46.98-30.23:.2f}   pop +{26.61-22.90:.2f}   pooled +{36.90-26.60:.2f}")
print(f"\nTotal (faithful@2.5 -> final): pooled {26.60:.1f} -> {47.01:.1f}  (+{47.01-26.60:.1f})")


In [ ]:
# (left) faithful @3.1 vs final router by split; (right) pooled-val climb across winners
splits = ["grouped", "pop", "pooled"]
fa = [46.98, 26.61, 36.90]; fin = [55.57, 38.27, 47.01]
x = np.arange(len(splits)); w = 0.38
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))

ax1.bar(x - w/2, fa, w, label="faithful (baseline)", color="#C44E52")
ax1.bar(x + w/2, fin, w, label="final router", color="#55A868")
ax1.set_xticks(x); ax1.set_xticklabels(splits); ax1.set_ylabel("val SimBench S")
ax1.set_title("faithful vs final system (held-out val)"); ax1.legend()
ax1.axhline(0, color="0.6", ls="--", lw=1)
for xi, (a, b) in enumerate(zip(fa, fin)):
    ax1.annotate(f"{a:.1f}", (xi - w/2, a), ha="center", va="bottom", fontsize=8)
    ax1.annotate(f"{b:.1f}", (xi + w/2, b), ha="center", va="bottom", fontsize=8)

climb = val_lineage.set_index("step")["pooled"]
ax2.plot(range(len(climb)), climb.values, marker="o", color="steelblue")
ax2.set_xticks(range(len(climb))); ax2.set_xticklabels(climb.index, rotation=25, ha="right")
ax2.set_ylabel("pooled val SimBench S"); ax2.set_title("Val-confirmed lineage (pooled climb)")
ax2.axhline(0, color="0.6", ls="--", lw=1)
for xi, v in enumerate(climb.values):
    ax2.annotate(f"{v:.1f}", (xi, v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()


---
## Pipeline validation — do we reproduce the paper?

`faithful` is the SimBench baseline prompt byte-for-byte. To check the whole
harness (prompt → parse → Eq. 2 normalization), run it on a model the paper
actually evaluated — **Qwen2.5-72B-Instruct** (paper Table 1: **S = 27.61**,
averaged across the pop + grouped splits) — and compare.

The paper tested 45 LLMs; Claude-3.7-Sonnet was best at 40.80. It predates and
does not include Gemini Flash-Lite, so our `gemini-3.1` numbers have no paper row
— Qwen2.5-72B is the shared anchor that validates the measurement.


In [ ]:
# faithful @ Qwen2.5-72B-Instruct — our pipeline on a model the paper evaluated.
# Stratified benchmark sample (pop 1442 + grouped 400); 24 transient API nulls excluded.
repro = pd.DataFrame([
    ["paper (Hu et al., Table 1)",             None,  None,  27.61],
    ["our pipeline (faithful @ Qwen2.5-72B)",  29.13, 24.53, 26.83],
], columns=["source", "S_grouped", "S_pop", "S (split-avg)"])
display(repro.set_index("source"))
print("our split-averaged S = 26.83, 95% CI [24.39, 29.35]")
print("paper reports 27.61  ->  inside our CI  ->  pipeline reproduces SimBench  ✓")


---
## Final headline — one-shot TEST (full lineage)

`test` was sealed throughout: every stage tuned on `dev`, and `val` was touched
only at gated confirmations (Stages 05 / 12 / 16). The table below is the one-shot
test evaluation of the whole lineage — both baseline models and every
val-confirmed winner — measured in a single pass. The final system was fixed on
`val` *before* any test contact, so this is **completeness measurement, not
selection-on-test**. Required questions (`trust_president`, `gay_rights`,
`internet_use`) are pinned to test.

Two honest takeaways:

1. **The model is the dominant lever.** faithful 2.5 → 3.1-flash-lite is **+15.9
   overall / +25.4 grouped** on test (required questions: −4.5 → +51.5), dwarfing
   the **+5.5** the entire method stack adds on top.
2. **On test, the task-kind router and the simpler `cc+abstain` are tied.** The
   router's val advantage (+1.1 pooled, Stage 16) did **not** transfer —
   `cc+abstain` is marginally ahead on overall/grouped/required, the router on pop,
   all within noise. Per leakage discipline we keep the val-confirmed router as the
   system, but the honest read is that `cc+abstain` is its equal here for far less
   machinery.


In [ ]:
# Recorded one-shot TEST scores — full lineage (both baseline models + every winner).
# Runs: outputs/runs/2026-06-21-TEST-{final,lineage,faithful-models}.results.json
test_lineage = pd.DataFrame([
    ["faithful",                        "2.5-flash-lite", 19.27, 13.77, 25.28, -4.51],
    ["faithful",                        "Qwen2.5-72B",    25.40, 20.97, 29.10, -1.26],
    ["faithful",                        "3.1-flash-lite", 35.21, 39.19, 30.88, 51.48],
    ["anti_flattening",                 "3.1-flash-lite", 39.68, 42.69, 36.39, 51.28],
    ["calibrated_commitment + abstain", "3.1-flash-lite", 40.93, 44.02, 37.55, 55.29],
    ["task-kind router (val-confirmed)","3.1-flash-lite", 40.73, 43.45, 37.77, 54.14],
], columns=["system", "model", "overall", "grouped", "pop", "required_Qs"])
display(test_lineage.set_index(["system", "model"]))

fig, ax = plt.subplots(figsize=(11, 4.6))
labels = [f"{s}\n({m})" for s, m in zip(test_lineage["system"], test_lineage["model"])]
x = np.arange(len(labels))
for col in ["overall", "grouped", "pop"]:
    ax.plot(x, test_lineage[col], marker="o", label=col)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("TEST SimBench S"); ax.axhline(0, color="0.6", ls="--", lw=1)
ax.set_title("One-shot TEST — full lineage (model lever >> method lever)")
ax.legend(); plt.tight_layout(); plt.show()

print("MODEL lever (faithful 2.5 -> 3.1):  +15.9 overall / +25.4 grouped.")
print("METHOD lever (faithful@3.1 -> final): +5.5 overall (shared by cc+abstain and the router).")
print("Router vs cc+abstain on TEST: TIED — cc+abstain +0.20 overall, +0.57 grouped, "
      "+1.15 required; router +0.22 pop. All within noise.")
print("-> The simpler cc+abstain is the deployable system; the router's val edge did not transfer.")


---
## Roads not taken — the negative results ledger

The lineage above is the **surviving** path. Many more ideas were pursued on `dev`
(a few to gated `val`) and **rejected** — the project is as much about what *didn't*
work as what did. They're catalogued here so the search is auditable and the
dead-ends don't get silently re-walked. The most thoroughly explored is **Stage 18**
(the grounded persona electorate — see `02_persona_factory.ipynb` §7 for its full
round-by-round story).

In [ ]:
# Roads not taken — dead-ends explored on dev/val and rejected (NOT in the final
# system). Recorded so the search is auditable and we don't re-walk them. Each has
# a full write-up in docs/experiments/stage-NN-*.md.
roads_not_taken = pd.DataFrame([
    ["03", "persona Monte-Carlo / diversity-CoT", "synthetic individuals over-disperse; averaging hedged draws stays hedged", "lost (pop 5.0, worst system)"],
    ["06", "post-hoc calibration (temp / entropy-temp / Dirichlet)", "correct over-confidence after the fact", "all < identity"],
    ["07", "superforecaster / CoT prompting", "make the model reason in steps before committing", "all < the no-CoT direct ask"],
    ["08", "entropy de-compression", "recover the missing spread from the prediction's own entropy", "no recoverable signal; global temps hurt"],
    ["09", "feature-predicted entropy", "predict truth-entropy from features, then temper", "R²=0.58 but still < identity (gain gated behind mode)"],
    ["13", "global discrete-vote simulation", "dispositional voters, vote tally = distribution", "refuted (1/3 datasets); only a narrow Choices13k win"],
    ["18", "grounded persona electorate (Nemotron)", "census-grounded bottom-up electorate + worldview enrichment", "best 51.8 < champion 63.2; ensemble +0.47 (noise)"],
], columns=["stage", "approach", "idea", "verdict"])
display(roads_not_taken.set_index("stage"))

print("Common thread: bottom-up simulation, elicited reasoning, and post-hoc")
print("recalibration all lose to the DIRECT distributional ask (+ selective abstention).")
print("The binding constraint is location/mode knowledge — not spread, not framing —")
print("which none of these approaches actually fix. What DID survive (Stages 10-16):")
print("a sharper one-call prompt (calibrated_commitment), abstaining where the model")
print("is below uniform, and task-context — all cheap, all targeting location.")

---
## Per-record diagnostics — `best_analysis_run()`

For repeatable per-record analysis of the final system (e.g. the error
decomposition in notebook 04), load the canonical run from the registry instead
of hardcoding a path. `best_analysis_run()` resolves a **dev** run by design — val
is gated and test is sealed, so neither is wired into a helper that runs on every
analysis pass. The registry note records the val↔test caveat.


In [ ]:
from scrye.results import best_analysis_run, ANALYSIS_RUNS

meta = ANALYSIS_RUNS["final"]
print(meta["description"])
print("note:", meta["note"])
per_system = json.load(open(best_analysis_run("final")))   # dev run; faithful vs final_router
print("systems:", list(per_system))

# final-system DEV score by dataset — where the per-kind routing acts
dev_by_ds = ev.score_by(pd.DataFrame(per_system["final_router"]), "dataset")
display(dev_by_ds.sort_values("mean_score", ascending=False).round(1).reset_index(drop=True))


---
## Ask the system a new question + demographic

`ask_winner(question, options, segment=..., year=...)` predicts an answer
distribution for a brand-new item. One live LLM call.

The final system is the **task-kind router**, but routing needs an *instrument
context* (the dataset's other items) that a one-off question doesn't have — so
for an ad-hoc item the router reduces to its base conditioning,
**`calibrated_commitment`** on `gemini-3.1-flash-lite`, which is what this helper
runs.


In [ ]:
from scrye.data import SimBenchRecord
from scrye.experiment import build_pipeline

def ask_winner(question, options, segment=None, year=None,
               model="gemini-3.1-flash-lite", strategy="calibrated_commitment", show=True):
    """Predict an answer distribution for a NEW question + demographic.

    options : list of "A) text" strings, or bare labels.
    segment : dict, e.g. {"country": "Finland", "age_group": "30-49", "gender": "female"}.
    """
    seg = dict(segment or {})
    if options and ")" in str(options[0]):
        labels = [str(o).split(")")[0].strip() for o in options]
        body = "\n".join(str(o) for o in options)
    else:
        labels = [str(o) for o in options]
        body = "\n".join(f"{l}) {l}" for l in labels)
    gp = []
    if year:
        gp.append(f"The year is {year}.")
    country = seg.get("country") or seg.get("cntry")
    if country:
        gp.append(f"You are from {country}.")
    rec = SimBenchRecord("ad_hoc", "grouped", f"{question.strip()}\n{body}",
                         tuple(labels), {l: 1 / len(labels) for l in labels},
                         " ".join(gp), segment=seg, num_grouping_vars=len(seg))
    dist = build_pipeline(model=model, predictor=strategy).predict(rec)
    if show:
        fig, ax = plt.subplots(figsize=(6, 0.5 * len(labels) + 1.2))
        ax.barh(list(dist), list(dist.values()), color="#55A868")
        ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("predicted probability")
        ax.set_title(f"{strategy} @ {model}\n{seg or 'general population'}")
        for l, v in dist.items():
            ax.annotate(f"{v:.2f}", (v, l), xytext=(3, 0),
                        textcoords="offset points", va="center", fontsize=9)
        plt.tight_layout(); plt.show()
    return dist


# Example — edit freely:
_ = ask_winner(
    "How important is it for you to live in a democratically governed country?",
    ["A) Not important", "B) Somewhat important", "C) Very important"],
    segment={"country": "Finland", "age_group": "65+"}, year=2024,
)


---
## Required-question predicted distributions (sealed test)

The three assignment-required questions — **`trust_president`**, **`gay_rights`**,
**`internet_use`** — are pinned to the **test** split, so they stayed sealed
through development. Below we render the model's *predicted* answer distribution
**Q** against the *human* distribution **P** for each, under the final reported
system.

This **visualizes the already-reported test result** (Stage 17; required-Q
S = 55.29), not selection-on-test. On these grouped opinion items the task-kind
router and the `AbstainCalibrator` both reduce to the base conditioning —
`calibrated_commitment` @ `gemini-3.1-flash-lite` — so that is what we run; every
call is served from the on-disk cache, so the pooled per-record mean S below
reproduces the recorded **55.29** exactly. Each question's distribution is pooled
across its segment variants, weighted by group size, into one P and one Q over
the answer options.

In [ ]:
from scrye.data import load_all
from scrye.splits import make_split

# Reproduce the sealed split — the three required questions are auto-pinned to `test`.
allr = load_all()
full = allr["grouped"] + allr["pop"]
split = make_split(full, seed=0, unit="question")
norms = ev.build_normalizers(full)                    # Eq.2 denominator on the FULL split
req_recs = ev.required_question_records(split.test)

# Final reported system. cc+abstain reduces to calibrated_commitment on these
# opinion items (abstain fires only on the failing behavioral datasets), so this
# IS the final system's behaviour here. All calls are cached -> free + deterministic.
final = build_pipeline(model="gemini-3.1-flash-lite", predictor="calibrated_commitment")

req_results = {q: ev.evaluate(final, recs, normalizers=norms, progress=False)
               for q, recs in req_recs.items()}
for q, rdf in req_results.items():
    print(f"{q:>16}: n={len(rdf):>3}  mean SimBench S = {rdf['score'].mean():6.2f}")
all_req = pd.concat(req_results.values(), ignore_index=True)
print(f"{'pooled':>16}: n={len(all_req):>3}  mean SimBench S = {all_req['score'].mean():6.2f}"
      f"   (matches the Stage-17 test report: 55.29)")

req_fig = viz.plot_required_question_distributions(req_results)
req_fig

### Save the headline figures to `outputs/visualizations/`

The deliverable-grade plots, written to `outputs/visualizations/` (gitignored
scratch) for reuse in write-ups: the required-question distributions above, the
Stage-01 conditioning ablation, the Stage-02 model-portability sweep, and the
one-shot TEST lineage.

In [ ]:
VIS_DIR = OUTPUTS_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

# 1) the Part-I deliverable: predicted vs human distributions for the required Qs
req_fig.savefig(VIS_DIR / "required_question_distributions.png", dpi=140, bbox_inches="tight")

# 2) Stage 01 — conditioning-strategy ablation (rebuilt from the persistent table t1)
viz.plot_system_comparison(t1, label_col="system").savefig(
    VIS_DIR / "ablation_strategy.png", dpi=140, bbox_inches="tight")

# 3) Stage 02 — model portability sweep
_f = Figure(figsize=(8, 4)); _ax = _f.subplots()
_sys = piv.index.tolist(); _x = np.arange(len(_sys)); _w = 0.25
for _i, _m in enumerate(piv.columns):
    _ax.bar(_x + (_i - 1) * _w, piv[_m].values, _w, label=_m.replace("gemini-", ""))
_ax.set_xticks(_x); _ax.set_xticklabels(_sys)
_ax.set_ylabel("grouped SimBench S"); _ax.axhline(0, color="0.6", ls="--", lw=1)
_ax.set_title("Stage 02 — model portability (grouped)"); _ax.legend(fontsize=8)
_f.tight_layout(); _f.savefig(VIS_DIR / "model_portability.png", dpi=140, bbox_inches="tight")

# 4) Final headline — one-shot TEST lineage (rebuilt from test_lineage)
_f = Figure(figsize=(11, 4.6)); _ax = _f.subplots()
_labels = [f"{s}\n({m})" for s, m in zip(test_lineage["system"], test_lineage["model"])]
_x = np.arange(len(_labels))
for _col in ["overall", "grouped", "pop", "required_Qs"]:
    _ax.plot(_x, test_lineage[_col], marker="o", label=_col)
_ax.set_xticks(_x); _ax.set_xticklabels(_labels, rotation=20, ha="right", fontsize=8)
_ax.set_ylabel("TEST SimBench S"); _ax.axhline(0, color="0.6", ls="--", lw=1)
_ax.set_title("One-shot TEST — full lineage"); _ax.legend(fontsize=8)
_f.tight_layout(); _f.savefig(VIS_DIR / "test_lineage.png", dpi=140, bbox_inches="tight")

print("saved 4 figures to", VIS_DIR)